# 90 · 3사 비교

입력: `evidence/<provider>/02_call.json`, `03_usage.json`, `05_costs.json` (노트북 10/20/30 또는 `python -m apicompare.demo` 가 생성) + `data/usage_log.jsonl` (usagelog).

이 노트북은 **API 를 호출하지 않는다.** evidence 가 없는 Provider 는 BLOCKED 행으로 표시된다. 표를 문서에 옮길 때는 숫자만 옮기고 응답 원문은 옮기지 않는다.

In [ ]:
import pandas as pd
from apicompare import PROVIDERS, PROVIDER_LABEL
from apicompare.evidence import load_evidence, badge


def dig(d, *paths, default=None):
    """여러 후보 경로('a.b.c') 중 처음 값이 있는 것을 돌려준다. evidence payload 모양이 노트북마다 조금 달라도 읽기 위함."""
    for path in paths:
        cur = d
        for key in path.split('.'):
            cur = cur.get(key) if isinstance(cur, dict) else None
            if cur is None:
                break
        if cur is not None:
            return cur
    return default


rows = []
for p in PROVIDERS:
    ev = load_evidence(p)
    call, usage, costs = ev.get('02_call'), ev.get('03_usage'), ev.get('05_costs')
    row = {'provider': PROVIDER_LABEL[p], 'model': None, 'latency_ms': None,
           'input': None, 'output': None, 'reasoning': None, 'total': None, 'usd': None,
           'badge': badge(p, '02_call')}
    if call is None and usage is None:
        row['badge'] = badge(p, '02_call')  # [BLOCKED: no evidence]
        rows.append(row)
        continue
    src = call or {}
    row['model'] = src.get('model') or dig(src, 'payload.model') or (usage or {}).get('model')
    row['latency_ms'] = dig(src, 'payload.latency_ms', 'payload.call.latency_ms')
    u = dig(usage or {}, 'payload.usage') or dig(src, 'payload.usage', 'payload.call.usage') or {}
    row['input'] = u.get('input_tokens')
    row['output'] = u.get('output_tokens')
    row['reasoning'] = u.get('reasoning_tokens')
    row['total'] = u.get('total_tokens')
    row['usd'] = dig(costs or {}, 'payload.local_estimate.usd', 'payload.cost.usd', 'payload.usd')
    if call is not None and call.get('source') == 'blocked':
        row['badge'] = badge(p, '02_call')
    rows.append(row)

df_compare = pd.DataFrame(rows)
df_compare

## usagelog 집계 (④ Usage 조회의 3사 공통 경로)

기간별 Usage API 는 OpenAI/Anthropic 은 Admin 키, Gemini 는 없음 → 호출마다 응답 usage 를 자체 적재해 집계한다. `data/usage_log.jsonl` 은 gitignore 대상.

In [ ]:
import pandas as pd
from apicompare import usagelog
from apicompare.models import to_dict

report = usagelog.aggregate(group_by=('provider', 'model'))
print(report.note)
df_usage = pd.DataFrame([
    {**b['group'], 'requests': b['requests'], 'input_tokens': b['input_tokens'],
     'output_tokens': b['output_tokens'], 'cost_usd': b['cost_usd'], 'start': b['start'][:19], 'end': b['end'][:19]}
    for b in to_dict(report.buckets)
])
df_usage if len(df_usage) else print('usage_log 비어 있음 — 노트북 10/20/30 의 ⑤ 섹션 또는 python -m apicompare.demo 가 적재한다')

## evidence 배지 3×7

In [ ]:
import pandas as pd
from apicompare import PROVIDERS, PROVIDER_LABEL
from apicompare.evidence import summary

table = {}
for p in PROVIDERS:
    for row in summary(p):
        table.setdefault(row['item_ko'], {})[PROVIDER_LABEL[p]] = row['badge']
df_badges = pd.DataFrame(table).T[[PROVIDER_LABEL[p] for p in PROVIDERS]]
df_badges

다음 단계: `uv run python scripts/build_comparison.py` 로 `docs/COMPARISON.md` 를 다시 생성하면 §8 evidence 현황에 위 배지가 반영된다.